# Reproducing, from scratch: *Social attention adds no predictive power for intraday move size beyond price volatility*

This notebook **re-implements the method step by step** from the raw dataset fields - it does not call a pre-written script.

The companion note showed attention spikes *precede* larger moves. Here we test whether attention adds anything a plain **price-volatility** reading does not. Crucially, **attention is not just a head count**: we measure it four ways - distinct authors, engagement (likes + retweets), reach (follower sum), and the **Engagement Coefficient** the Instrumetriq extension shows as a coin's chatter level - and check every one. The measure that decides usefulness is **ranking skill (AUC)**: the chance a reading ranks a real large-mover above a non-mover (0.50 = a coin flip). Everything is measured **out of sample**.

Uses only the **free weekly samples**. No purchase or private data required.

## 0. Setup

In [ ]:
!pip -q install pandas pyarrow numpy scikit-learn
import glob
import numpy as np
import pandas as pd

## 1. Get the sample data
An anonymous read of a public repo - no login needed.

In [ ]:
!git clone --depth 1 https://github.com/SiCkGFX/instrumetriq-public.git 2>/dev/null || (cd instrumetriq-public && git pull -q)
FILES = sorted(glob.glob('instrumetriq-public/samples/week_*/*_tier3.parquet'))
print(len(FILES), 'weekly Tier 3 sample files')

## 2. What we read from each session
Each row is one coin over one ~2-hour session. The attention inputs all live in `twitter_sentiment_windows.last_cycle`, fixed at admission (before the price path that follows):

- **authors** - `author_stats.distinct_authors_total`
- **engagement** - `platform_engagement.total_likes` + `total_retweets`
- **reach** - `author_stats.followers_count_sum`
- **posts** - `posts_total`

Plus **price volatility** `spot_raw.range_pct_24h` (24h range at admission, price-only) and the **price path** `spot_prices[].mid` (the outcome).

In [ ]:
df0 = pd.read_parquet(FILES[0], columns=['symbol','twitter_sentiment_windows','spot_raw','spot_prices'])
lc = df0.iloc[0]['twitter_sentiment_windows'].get('last_cycle')
astat = lc.get('author_stats') or {}; eng = lc.get('platform_engagement') or {}
print('distinct_authors_total :', astat.get('distinct_authors_total'))
print('likes, retweets        :', eng.get('total_likes'), eng.get('total_retweets'))
print('followers_count_sum    :', astat.get('followers_count_sum'))
print('posts_total            :', lc.get('posts_total'))
print('range_pct_24h          :', df0.iloc[0]['spot_raw'].get('range_pct_24h'), ' <- price volatility')

## 3. Build the four attention measures and the outcome
For each session we compute the four attention inputs, including the **Engagement Coefficient** exactly as the extension defines it:

`EC = (likes + retweets) * log(1 + followers) / posts`

The outcome is the forward **move magnitude** = `max |mid/m0 - 1|` over the price path (mid of every 3rd tick, m0 the first).

In [ ]:
def extract(files):
    rows = []
    for fp in files:
        df = pd.read_parquet(fp, columns=['symbol','snapshot_ts',
                'twitter_sentiment_windows','spot_raw','spot_prices'])
        for sym, ts, tsw, sr, sp in zip(df['symbol'].values, df['snapshot_ts'].values,
                df['twitter_sentiment_windows'].values, df['spot_raw'].values, df['spot_prices'].values):
            lc = tsw.get('last_cycle') if hasattr(tsw,'get') else None
            if lc is None: continue
            posts  = lc.get('posts_total')
            silent = (lc.get('sentiment_activity') or {}).get('is_silent')
            astat  = lc.get('author_stats') or {}; eng = lc.get('platform_engagement') or {}
            authors = astat.get('distinct_authors_total'); reach = astat.get('followers_count_sum')
            engagement = (eng.get('total_likes') or 0) + (eng.get('total_retweets') or 0)
            vol24 = sr.get('range_pct_24h') if hasattr(sr,'get') else None
            if not posts or silent or authors is None or vol24 is None: continue
            if sp is None or len(sp) < 6: continue
            mids = [float(s['mid']) for s in sp[::3] if s.get('mid')]
            if len(mids) < 5 or not mids[0]: continue
            a = np.asarray(mids)
            EC = engagement * np.log1p(reach or 0) / posts          # the extension's Engagement Coefficient
            rows.append((sym, pd.Timestamp(ts), float(authors), float(engagement),
                         float(reach or 0), float(EC), float(vol24), float(np.max(np.abs(a/a[0]-1.0)))))
    d = pd.DataFrame(rows, columns=['symbol','ts','authors','engagement','reach','EC','vol24','maxabs'])
    d['ts'] = pd.to_datetime(d['ts'], utc=True); d['day'] = d['ts'].dt.strftime('%Y-%m-%d')
    return d.sort_values(['symbol','ts'])

d = extract(FILES)
print(f'{len(d):,} sessions with observable social activity')

## 4. Attention spike = measure / the coin's own recent baseline
Each attention measure is judged relative to the coin's own history: a past-only median of prior sessions (`shift(1)` excludes the current one). The weekly samples are one day per week, so we use an intraday baseline (earlier same-day sessions).

In [ ]:
MEASURES = ['authors','engagement','reach','EC']
for m in MEASURES:
    base = d.groupby(['symbol','day'])[m].transform(lambda s: s.shift(1).expanding(min_periods=5).median())
    d[m+'_spike'] = np.where(base > 0, d[m]/base, np.nan)
d = d[np.isfinite(d['authors_spike'])].copy()
print(f'{len(d):,} sessions with a computable baseline')

## 5. Reproduce the companion note (its measure: author-count spike)
The companion note defined the spike on author count, so we reproduce its lift on that measure.

In [ ]:
for thr in (0.03, 0.04, 0.05):
    h = d['maxabs'].values >= thr
    s = d['authors_spike'].values >= 6.0
    n = (d['authors_spike'].values >= 0.8) & (d['authors_spike'].values <= 1.2)
    print(f'|move|>={thr:.0%}: spike {h[s].mean():5.1%} vs normal {h[n].mean():5.1%}  lift={h[s].mean()/h[n].mean():.2f}x')

## 6. Hold out later days; ranking skill and added value for every attention measure
Hold out the later days. For each attention measure, measure its **ranking skill** (AUC on the held-out set) and its **added value** over a volatility-only model (out of sample). A predictor that helps would raise AUC above volatility's; a redundant one leaves it unchanged.

In [ ]:
from sklearn.linear_model import LogisticRegression
days = np.sort(d['day'].unique()); cut = days[int(len(days)*0.70)]
train = d['day'].values < cut; test = d['day'].values >= cut
y = (d['maxabs'].values >= 0.04).astype(float)          # large move = |move| >= 4%

def auc(score, label):
    score=np.asarray(score,float); label=np.asarray(label,float)
    ok=~np.isnan(score); score,label=score[ok],label[ok]; n1,n=label.sum(),len(label)
    if n1==0 or n1==n: return None
    o=np.argsort(score,kind='mergesort'); r=np.empty(n); r[o]=np.arange(1,n+1)
    return float((r[label==1].sum()-n1*(n1+1)/2)/(n1*(n-n1)))
def oos_auc(X):
    Xtr,Xte=X[train],X[test]; med=np.nanmedian(Xtr,0)
    Xtr=np.where(np.isnan(Xtr),med,Xtr); Xte=np.where(np.isnan(Xte),med,Xte)
    mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    m=LogisticRegression(max_iter=2000,class_weight='balanced').fit((Xtr-mu)/sd,y[train])
    return auc(m.predict_proba((Xte-mu)/sd)[:,1],y[test])

vol=d['vol24'].values; a_vol=oos_auc(vol[:,None])
print(f'price volatility ranks movers at AUC {a_vol:.3f}\n')
print(f'{"attention measure":18s}{"ranking skill (AUC)":>20}{"added over volatility":>22}')
for m in MEASURES:
    sp=d[m+'_spike'].values
    print(f'{m:18s}{auc(sp[test],y[test]):>20.3f}{oos_auc(np.column_stack([vol,np.log1p(d[m].values),sp]))-a_vol:>+22.3f}')

## 7. Sanity check: a positive control
Swap attention for a feature that peeks at the answer (outcome + noise). The added value must jump - proving the test can see real signal.

In [ ]:
g=np.random.default_rng(0)
for noise in (0.5,2.0):
    cheat=y+g.normal(0,noise,len(y))
    print(f'noise={noise}: volatility {a_vol:.3f} -> volatility+cheat {oos_auc(np.column_stack([vol,cheat])):.3f}   '
          f'added {oos_auc(np.column_stack([vol,cheat]))-a_vol:+.3f}')

## 8. Result
- Attention **precedes** larger moves (author-spike lift ~2.4x) - the companion note holds.
- But **every** attention measure - authors, engagement, reach, and the extension's Engagement Coefficient - ranks movers near a **coin flip** (~0.48-0.51) and adds **~0.00** to price volatility, while the peeking control adds a lot.

**Attention, by any measure, is a shadow of volatility - not an independent predictor of intraday move size.** On the full contiguous archive the same holds with tighter numbers (`--mode trailing`).